# Healthcare Readmission Analytics
## 01 — Raw Data Audit

### Project Objective
Analyze inpatient diabetes encounters from 130 U.S. hospitals to identify patterns associated with hospital readmission, with particular emphasis on early readmission within 30 days.

### Dataset
**Diabetes 130-US Hospitals for Years 1999–2008**

Source: UCI Machine Learning Repository

Primary file: `diabetic_data.csv`  
Reference mapping file: `IDS_mapping.csv`

### Primary Analytical Question
**Which patient, clinical, utilization, and encounter characteristics are associated with an increased likelihood of hospital readmission within 30 days?**

### Purpose of This Notebook
This notebook performs the initial audit of the raw source data before any cleaning, transformation, statistical modeling, or machine-learning development.

The audit will establish:

1. Dataset dimensions and schema
2. Encounter and patient uniqueness
3. Duplicate records and identifiers
4. Target-variable distribution
5. Missing-value patterns, including encoded missing values
6. Variable cardinality and data types
7. Repeated encounters per patient
8. Initial data-quality risks
9. Potential leakage variables
10. Requirements for the downstream cleaning and PostgreSQL pipeline

### Data Integrity Principle
Files stored under `data/raw/` are treated as immutable source data. All cleaning and transformations will produce separate downstream datasets.

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

print("python executable:", sys.executable)
print("python version:", sys.version.split()[0])
print("numpy version:", np.__version__)
print("pandas version:", pd.__version__)
print("matplotlib version:", matplotlib.__version__)

python executable: c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\.venv\Scripts\python.exe
python version: 3.13.6
numpy version: 2.5.2
pandas version: 3.0.5
matplotlib version: 3.11.1


In [3]:
# Define project-relative paths
PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

DIABETIC_DATA_PATH = RAW_DATA_DIR / "diabetic_data.csv"
IDS_MAPPING_PATH = RAW_DATA_DIR / "IDS_mapping.csv"

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", RAW_DATA_DIR)

print("\nSource file checks:")
print("diabetic_data.csv exists:", DIABETIC_DATA_PATH.exists())
print("IDS_mapping.csv exists:", IDS_MAPPING_PATH.exists())

Project root: c:\Users\amohe.000\Downloads\healthcare-readmission-analytics
Raw data directory: c:\Users\amohe.000\Downloads\healthcare-readmission-analytics\data\raw

Source file checks:
diabetic_data.csv exists: True
IDS_mapping.csv exists: True


In [4]:
# Load the primary raw dataset without modifying the source file
df_raw = pd.read_csv(DIABETIC_DATA_PATH)

print("Dataset loaded successfully.")
print(f"Rows: {df_raw.shape[0]:,}")
print(f"Columns: {df_raw.shape[1]}")
print(f"Total cells: {df_raw.size:,}")

Dataset loaded successfully.
Rows: 101,766
Columns: 50
Total cells: 5,088,300


In [5]:
# Now we want to answer a different question: What exactly did pandas load?

# Inspect raw dataset schema 
schema = pd.DataFrame(
    {"column": df_raw.columns, "dtype": df_raw.dtypes.astype(str).values}
)

print(f"Number of columns: {len(schema)}")
schema

Number of columns: 50


,column,dtype
0,encounter_id,int64
1,patient_nbr,int64
2,race,str
3,gender,str
4,age,str
5,weight,str
6,admission_type_id,int64
7,discharge_disposition_id,int64
8,admission_source_id,int64
9,time_in_hospital,int64


In [6]:
# Inspect the first five raw encounter records
pd.set_option("display.max_columns", None)

df_raw.head()

,encounter_id,patient_nbr,race,gender,age,weight,admission_type_id,discharge_disposition_id,admission_source_id,time_in_hospital,payer_code,medical_specialty,num_lab_procedures,num_procedures,num_medications,number_outpatient,number_emergency,number_inpatient,diag_1,diag_2,diag_3,number_diagnoses,max_glu_serum,A1Cresult,metformin,repaglinide,nateglinide,chlorpropamide,glimepiride,acetohexamide,glipizide,glyburide,tolbutamide,pioglitazone,rosiglitazone,acarbose,miglitol,troglitazone,tolazamide,examide,citoglipton,insulin,glyburide-metformin,glipizide-metformin,glimepiride-pioglitazone,metformin-rosiglitazone,metformin-pioglitazone,change,diabetesMed,readmitted
0,2278392,8222157,Caucasian,Female,[0-10),?,6,25,1,1,?,Pediatrics-Endocrinology,41,0,1,0,0,0,250.83,?,?,1,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,NO
1,149190,55629189,Caucasian,Female,[10-20),?,1,1,7,3,?,?,59,0,18,0,0,0,276,250.01,255,9,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,>30
2,64410,86047875,AfricanAmerican,Female,[20-30),?,1,1,7,2,?,?,11,5,13,2,0,1,648,250,V27,6,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Yes,NO
3,500364,82442376,Caucasian,Male,[30-40),?,1,1,7,2,?,?,44,1,16,0,0,0,8,250.43,403,7,NaN,NaN,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,No,Up,No,No,No,No,No,Ch,Yes,NO
4,16680,42519267,Caucasian,Male,[40-50),?,1,1,7,1,?,?,51,0,8,0,0,0,197,157,250,5,NaN,NaN,No,No,No,No,No,No,Steady,No,No,No,No,No,No,No,No,No,No,Steady,No,No,No,No,No,Ch,Yes,NO


In [7]:
# Audit explicit "?" missing-value markers in the raw dataset

question_mark_counts = (df_raw == "?").sum()

question_mark_audit = (
    pd.DataFrame(
        {
            "question_mark_count": question_mark_counts,
            "question_mark_pct": (question_mark_counts / len(df_raw) * 100).round(2),
        }
    )
    .query("question_mark_count > 0")
    .sort_values("question_mark_pct", ascending=False)
)

print(f'Columns containing "?": {len(question_mark_audit)}')
print(f'Total "?" values: {question_mark_audit["question_mark_count"].sum():,}')

question_mark_audit

Columns containing "?": 7
Total "?" values: 192,849


,question_mark_count,question_mark_pct
weight,98569,96.86
medical_specialty,49949,49.08
payer_code,40256,39.56
race,2273,2.23
diag_3,1423,1.40
diag_2,358,0.35
diag_1,21,0.02


In [8]:
# Audit native null / NaN values detected by pandas

native_null_counts = df_raw.isna().sum()

native_null_audit = (
    pd.DataFrame(
        {
            "null_count": native_null_counts,
            "null_pct": (native_null_counts / len(df_raw) * 100).round(2),
        }
    )
    .query("null_count > 0")
    .sort_values("null_pct", ascending=False)
)

print(f"Columns containing native nulls: {len(native_null_audit)}")
print(f"Total native null values: {native_null_audit['null_count'].sum():,}")

native_null_audit

Columns containing native nulls: 2
Total native null values: 181,168


,null_count,null_pct
max_glu_serum,96420,94.75
A1Cresult,84748,83.28


In [9]:
# Build a unified missingness audit:
# explicit "?" sentinels + native NaN values

question_mark_counts = (df_raw == "?").sum()
native_null_counts = df_raw.isna().sum()

missingness_audit = pd.DataFrame(
    {
        "question_mark_count": question_mark_counts,
        "native_null_count": native_null_counts,
    }
)

missingness_audit["total_missing_count"] = (
    missingness_audit["question_mark_count"] + missingness_audit["native_null_count"]
)

missingness_audit["total_missing_pct"] = (
    missingness_audit["total_missing_count"] / len(df_raw) * 100
).round(2)

missingness_audit = missingness_audit.query("total_missing_count > 0").sort_values(
    "total_missing_pct", ascending=False
)

print(f"Columns with identified missingness: {len(missingness_audit)}")
print(
    f"Total identified missing cells: "
    f"{missingness_audit['total_missing_count'].sum():,}"
)

missingness_audit

Columns with identified missingness: 9
Total identified missing cells: 374,017


,question_mark_count,native_null_count,total_missing_count,total_missing_pct
weight,98569,0,98569,96.86
max_glu_serum,0,96420,96420,94.75
A1Cresult,0,84748,84748,83.28
medical_specialty,49949,0,49949,49.08
payer_code,40256,0,40256,39.56
race,2273,0,2273,2.23
diag_3,1423,0,1423,1.40
diag_2,358,0,358,0.35
diag_1,21,0,21,0.02


In [10]:
# Audit encounter and patient identifier integrity

total_rows = len(df_raw)

unique_encounters = df_raw["encounter_id"].nunique()
unique_patients = df_raw["patient_nbr"].nunique()

duplicate_encounter_ids = df_raw["encounter_id"].duplicated().sum()

encounters_per_patient = (
    df_raw.groupby("patient_nbr").size().sort_values(ascending=False)
)

patients_with_multiple_encounters = (encounters_per_patient > 1).sum()
max_encounters_per_patient = encounters_per_patient.max()

print(f"Total encounter rows: {total_rows:,}")
print(f"Unique encounter IDs: {unique_encounters:,}")
print(f"Duplicate encounter IDs: {duplicate_encounter_ids:,}")
print()
print(f"Unique patients: {unique_patients:,}")
print(f"Patients with multiple encounters: {patients_with_multiple_encounters:,}")
print(
    f"Patients with multiple encounters (%): "
    f"{patients_with_multiple_encounters / unique_patients * 100:.2f}%"
)
print(f"Maximum encounters for one patient: {max_encounters_per_patient}")

Total encounter rows: 101,766
Unique encounter IDs: 101,766
Duplicate encounter IDs: 0

Unique patients: 71,518
Patients with multiple encounters: 16,773
Patients with multiple encounters (%): 23.45%
Maximum encounters for one patient: 40


In [11]:
# Audit the raw readmission target distribution

readmission_counts = df_raw["readmitted"].value_counts(dropna=False)

readmission_audit = pd.DataFrame(
    {
        "encounter_count": readmission_counts,
        "encounter_pct": (readmission_counts / len(df_raw) * 100).round(2),
    }
)

print("Raw readmission categories:")
print(f"Number of target categories: {df_raw['readmitted'].nunique(dropna=False)}")
print(f"Missing target values: {df_raw['readmitted'].isna().sum():,}")

readmission_audit

Raw readmission categories:
Number of target categories: 3
Missing target values: 0


,encounter_count,encounter_pct
readmitted,,
NO,54864,53.91
>30,35545,34.93
<30,11357,11.16


In [12]:
# Calculate 30-day readmission prevalence without modifying df_raw

readmitted_30d = df_raw["readmitted"].eq("<30")

positive_count = readmitted_30d.sum()
negative_count = (~readmitted_30d).sum()

positive_pct = positive_count / len(df_raw) * 100
negative_pct = negative_count / len(df_raw) * 100

print(f"30-day readmission encounters: {positive_count:,}")
print(f"30-day readmission rate: {positive_pct:.2f}%")
print()
print(f"Not readmitted within 30 days: {negative_count:,}")
print(f"Non-event rate: {negative_pct:.2f}%")
print()
print(f"Naive majority-class accuracy: {negative_pct:.2f}%")

30-day readmission encounters: 11,357
30-day readmission rate: 11.16%

Not readmitted within 30 days: 90,409
Non-event rate: 88.84%

Naive majority-class accuracy: 88.84%


In [13]:
# Audit exact duplicate rows across all columns

duplicate_rows = df_raw.duplicated().sum()

print(f"Exact duplicate rows: {duplicate_rows:,}")
print(f"Duplicate-row rate: {duplicate_rows / len(df_raw) * 100:.4f}%")

Exact duplicate rows: 0
Duplicate-row rate: 0.0000%


In [14]:
# Audit cardinality (number of distinct values) for every raw column

cardinality_audit = pd.DataFrame(
    {
        "dtype": df_raw.dtypes.astype(str),
        "unique_non_null": df_raw.nunique(dropna=True),
        "unique_including_null": df_raw.nunique(dropna=False),
    }
)

cardinality_audit["unique_pct"] = (
    cardinality_audit["unique_non_null"] / len(df_raw) * 100
).round(2)

cardinality_audit = cardinality_audit.sort_values(
    ["unique_non_null", "unique_pct"], ascending=[True, True]
)

print(f"Columns audited: {len(cardinality_audit)}")
cardinality_audit

Columns audited: 50


,dtype,unique_non_null,unique_including_null,unique_pct
examide,str,1,1,0.00
citoglipton,str,1,1,0.00
acetohexamide,str,2,2,0.00
tolbutamide,str,2,2,0.00
troglitazone,str,2,2,0.00
glipizide-metformin,str,2,2,0.00
glimepiride-pioglitazone,str,2,2,0.00
metformin-rosiglitazone,str,2,2,0.00
metformin-pioglitazone,str,2,2,0.00
change,str,2,2,0.00


In [15]:
# Inspect value distributions for low-cardinality columns

low_cardinality_cols = cardinality_audit[
    cardinality_audit["unique_non_null"] <= 10
].index.tolist()

print(
    f"Low-cardinality columns (<=10 unique non-null values): {len(low_cardinality_cols)}"
)
print(low_cardinality_cols)

for col in low_cardinality_cols:
    print("\n" + "=" * 70)
    print(f"COLUMN: {col}")
    print(f"DTYPE: {df_raw[col].dtype}")
    print(f"UNIQUE NON-NULL VALUES: {df_raw[col].nunique(dropna=True)}")
    print("-" * 70)

    print(df_raw[col].value_counts(dropna=False).to_string())

Low-cardinality columns (<=10 unique non-null values): 34
['examide', 'citoglipton', 'acetohexamide', 'tolbutamide', 'troglitazone', 'glipizide-metformin', 'glimepiride-pioglitazone', 'metformin-rosiglitazone', 'metformin-pioglitazone', 'change', 'diabetesMed', 'gender', 'max_glu_serum', 'A1Cresult', 'tolazamide', 'readmitted', 'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide', 'glimepiride', 'glipizide', 'glyburide', 'pioglitazone', 'rosiglitazone', 'acarbose', 'miglitol', 'insulin', 'glyburide-metformin', 'race', 'num_procedures', 'admission_type_id', 'age', 'weight']

COLUMN: examide
DTYPE: str
UNIQUE NON-NULL VALUES: 1
----------------------------------------------------------------------
examide
No    101766

COLUMN: citoglipton
DTYPE: str
UNIQUE NON-NULL VALUES: 1
----------------------------------------------------------------------
citoglipton
No    101766

COLUMN: acetohexamide
DTYPE: str
UNIQUE NON-NULL VALUES: 2
-----------------------------------------------------

In [16]:
# Build a compact audit of low-cardinality columns

low_cardinality_summary = []

for col in low_cardinality_cols:
    counts = df_raw[col].value_counts(dropna=False)

    low_cardinality_summary.append(
        {
            "column": col,
            "dtype": str(df_raw[col].dtype),
            "n_unique": df_raw[col].nunique(dropna=True),
            "most_common_value": counts.index[0],
            "most_common_count": counts.iloc[0],
            "most_common_pct": round(counts.iloc[0] / len(df_raw) * 100, 2),
        }
    )

low_cardinality_summary = (
    pd.DataFrame(low_cardinality_summary)
    .sort_values(["n_unique", "most_common_pct"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Low-cardinality columns summarized: {len(low_cardinality_summary)}")

low_cardinality_summary

Low-cardinality columns summarized: 34


,column,dtype,n_unique,most_common_value,most_common_count,most_common_pct
0,examide,str,1,No,101766,100.00
1,citoglipton,str,1,No,101766,100.00
2,acetohexamide,str,2,No,101765,100.00
3,troglitazone,str,2,No,101763,100.00
4,glimepiride-pioglitazone,str,2,No,101765,100.00
5,metformin-rosiglitazone,str,2,No,101764,100.00
6,metformin-pioglitazone,str,2,No,101765,100.00
7,glipizide-metformin,str,2,No,101753,99.99
8,tolbutamide,str,2,No,101743,99.98
9,diabetesMed,str,2,Yes,78363,77.00


In [17]:
# Audit numeric columns for ranges and suspicious values

numeric_cols = df_raw.select_dtypes(include=np.number).columns.tolist()

numeric_audit = pd.DataFrame(
    {
        "dtype": df_raw[numeric_cols].dtypes.astype(str),
        "non_null_count": df_raw[numeric_cols].count(),
        "unique_values": df_raw[numeric_cols].nunique(dropna=True),
        "min": df_raw[numeric_cols].min(),
        "median": df_raw[numeric_cols].median(),
        "mean": df_raw[numeric_cols].mean(),
        "max": df_raw[numeric_cols].max(),
    }
)

numeric_audit["zero_count"] = (df_raw[numeric_cols] == 0).sum()
numeric_audit["negative_count"] = (df_raw[numeric_cols] < 0).sum()

numeric_audit = numeric_audit.round(2)

print(f"Numeric columns audited: {len(numeric_cols)}")
numeric_audit

Numeric columns audited: 13


,dtype,non_null_count,unique_values,min,median,mean,max,zero_count,negative_count
encounter_id,int64,101766,101766,12522,152388987.0,1.652016e+08,443867222,0,0
patient_nbr,int64,101766,71518,135,45505143.0,5.433040e+07,189502619,0,0
admission_type_id,int64,101766,8,1,1.0,2.020000e+00,8,0,0
discharge_disposition_id,int64,101766,26,1,1.0,3.720000e+00,28,0,0
admission_source_id,int64,101766,17,1,7.0,5.750000e+00,25,0,0
time_in_hospital,int64,101766,14,1,4.0,4.400000e+00,14,0,0
num_lab_procedures,int64,101766,118,1,44.0,4.310000e+01,132,0,0
num_procedures,int64,101766,7,0,1.0,1.340000e+00,6,46652,0
num_medications,int64,101766,75,1,15.0,1.602000e+01,81,0,0
number_outpatient,int64,101766,39,0,0.0,3.700000e-01,42,85027,0


In [18]:
# Display the complete numeric audit

pd.set_option("display.max_rows", None)

numeric_audit

,dtype,non_null_count,unique_values,min,median,mean,max,zero_count,negative_count
encounter_id,int64,101766,101766,12522,152388987.0,1.652016e+08,443867222,0,0
patient_nbr,int64,101766,71518,135,45505143.0,5.433040e+07,189502619,0,0
admission_type_id,int64,101766,8,1,1.0,2.020000e+00,8,0,0
discharge_disposition_id,int64,101766,26,1,1.0,3.720000e+00,28,0,0
admission_source_id,int64,101766,17,1,7.0,5.750000e+00,25,0,0
time_in_hospital,int64,101766,14,1,4.0,4.400000e+00,14,0,0
num_lab_procedures,int64,101766,118,1,44.0,4.310000e+01,132,0,0
num_procedures,int64,101766,7,0,1.0,1.340000e+00,6,46652,0
num_medications,int64,101766,75,1,15.0,1.602000e+01,81,0,0
number_outpatient,int64,101766,39,0,0.0,3.700000e-01,42,85027,0


In [19]:
# Define semantic roles for numeric columns
# This does not modify the raw dataset.

identifier_cols = ["encounter_id", "patient_nbr"]

coded_categorical_cols = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
]

quantitative_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses",
]

classified_numeric_cols = identifier_cols + coded_categorical_cols + quantitative_cols

print("Numeric semantic-role audit")
print("-" * 50)
print(f"Identifiers:                {len(identifier_cols)}")
print(f"Coded categorical fields:   {len(coded_categorical_cols)}")
print(f"Quantitative/count fields:  {len(quantitative_cols)}")
print(f"Total classified:           {len(classified_numeric_cols)}")
print(f"Total numeric columns:       {len(numeric_cols)}")

print(
    "\nAll numeric columns classified:",
    set(classified_numeric_cols) == set(numeric_cols),
)

Numeric semantic-role audit
--------------------------------------------------
Identifiers:                2
Coded categorical fields:   3
Quantitative/count fields:  8
Total classified:           13
Total numeric columns:       13

All numeric columns classified: True


In [20]:
# Load and inspect the raw ID mapping reference file
ids_mapping_raw = pd.read_csv(IDS_MAPPING_PATH)

print("IDS mapping file loaded successfully.")
print(f"Rows: {ids_mapping_raw.shape[0]:,}")
print(f"Columns: {ids_mapping_raw.shape[1]:,}")
print("\nColumn names:")
print(ids_mapping_raw.columns.tolist())

ids_mapping_raw.head(20)

IDS mapping file loaded successfully.
Rows: 67
Columns: 2

Column names:
['admission_type_id', 'description']


,admission_type_id,description
0,1,Emergency
1,2,Urgent
2,3,Elective
3,4,Newborn
4,5,Not Available
5,6,NaN
6,7,Trauma Center
7,8,Not Mapped
8,NaN,NaN
9,discharge_disposition_id,description


In [21]:
# Inspect every row of the raw mapping file to identify embedded lookup sections
pd.set_option("display.max_rows", None)

ids_mapping_display = ids_mapping_raw.reset_index().rename(
    columns={"index": "row_number"}
)

ids_mapping_display

,row_number,admission_type_id,description
0,0,1,Emergency
1,1,2,Urgent
2,2,3,Elective
3,3,4,Newborn
4,4,5,Not Available
5,5,6,NaN
6,6,7,Trauma Center
7,7,8,Not Mapped
8,8,NaN,NaN
9,9,discharge_disposition_id,description


In [22]:
# Locate embedded lookup-table headers in IDS_mapping.csv

expected_mapping_headers = {
    "discharge_disposition_id",
    "admission_source_id",
}

mapping_header_rows = ids_mapping_raw[
    ids_mapping_raw["admission_type_id"].isin(expected_mapping_headers)
].copy()

mapping_header_rows = mapping_header_rows.reset_index().rename(
    columns={"index": "row_number"}
)

print("Embedded mapping headers detected:")
mapping_header_rows

Embedded mapping headers detected:


,row_number,admission_type_id,description
0,9,discharge_disposition_id,description
1,41,admission_source_id,description


In [23]:
# Parse the stacked IDS_mapping.csv file into three clean lookup tables

# 1. Admission type mappings
admission_type_lookup = (
    ids_mapping_raw.iloc[0:8]
    .rename(
        columns={
            "admission_type_id": "admission_type_id",
            "description": "admission_type_description",
        }
    )
    .copy()
)

# 2. Discharge disposition mappings
discharge_disposition_lookup = (
    ids_mapping_raw.iloc[10:41]
    .rename(
        columns={
            "admission_type_id": "discharge_disposition_id",
            "description": "discharge_disposition_description",
        }
    )
    .copy()
)

# 3. Admission source mappings
admission_source_lookup = (
    ids_mapping_raw.iloc[42:67]
    .rename(
        columns={
            "admission_type_id": "admission_source_id",
            "description": "admission_source_description",
        }
    )
    .copy()
)

print("Lookup tables created:")
print(f"Admission types:          {len(admission_type_lookup)} rows")
print(f"Discharge dispositions:  {len(discharge_disposition_lookup)} rows")
print(f"Admission sources:        {len(admission_source_lookup)} rows")

Lookup tables created:
Admission types:          8 rows
Discharge dispositions:  31 rows
Admission sources:        25 rows


In [24]:
# Audit lookup-table coverage against codes actually used in df_raw

lookup_pairs = {
    "admission_type_id": admission_type_lookup,
    "discharge_disposition_id": discharge_disposition_lookup,
    "admission_source_id": admission_source_lookup,
}

coverage_results = []

for column, lookup in lookup_pairs.items():

    # Convert lookup IDs to numeric for valid comparison with df_raw
    lookup_codes = pd.to_numeric(lookup[column], errors="coerce").dropna().astype(int)

    dataset_codes = set(df_raw[column].dropna().unique())
    mapped_codes = set(lookup_codes.unique())

    unmapped_codes = sorted(dataset_codes - mapped_codes)

    unmapped_encounters = df_raw[column].isin(unmapped_codes).sum()

    coverage_results.append(
        {
            "column": column,
            "dataset_unique_codes": len(dataset_codes),
            "mapped_unique_codes": len(dataset_codes & mapped_codes),
            "unmapped_unique_codes": len(unmapped_codes),
            "unmapped_encounters": unmapped_encounters,
            "unmapped_codes": unmapped_codes,
        }
    )

lookup_coverage_audit = pd.DataFrame(coverage_results)

lookup_coverage_audit

,column,dataset_unique_codes,mapped_unique_codes,unmapped_unique_codes,unmapped_encounters,unmapped_codes
0,admission_type_id,8,8,0,0,[]
1,discharge_disposition_id,26,26,0,0,[]
2,admission_source_id,17,17,0,0,[]


In [25]:
# Build human-readable frequency audits for coded categorical fields

coded_lookup_pairs = {
    "admission_type_id": (admission_type_lookup, "admission_type_description"),
    "discharge_disposition_id": (
        discharge_disposition_lookup,
        "discharge_disposition_description",
    ),
    "admission_source_id": (admission_source_lookup, "admission_source_description"),
}

coded_distribution_audits = {}

for column, (lookup, description_col) in coded_lookup_pairs.items():

    # Prepare a temporary numeric lookup — original lookup remains unchanged
    lookup_temp = lookup.copy()
    lookup_temp[column] = pd.to_numeric(lookup_temp[column], errors="coerce")
    lookup_temp = lookup_temp.dropna(subset=[column])
    lookup_temp[column] = lookup_temp[column].astype(int)

    # Count codes appearing in the encounter dataset
    audit = (
        df_raw[column]
        .value_counts(dropna=False)
        .rename_axis(column)
        .reset_index(name="encounter_count")
    )

    audit["encounter_pct"] = (audit["encounter_count"] / len(df_raw) * 100).round(2)

    # Attach human-readable descriptions
    audit = audit.merge(
        lookup_temp[[column, description_col]],
        on=column,
        how="left",
        validate="many_to_one",
    )

    audit = audit[[column, description_col, "encounter_count", "encounter_pct"]]

    coded_distribution_audits[column] = audit

    print("\n" + "=" * 80)
    print(column.upper())
    print("=" * 80)
    print(audit.to_string(index=False))


ADMISSION_TYPE_ID
 admission_type_id admission_type_description  encounter_count  encounter_pct
                 1                  Emergency            53990          53.05
                 3                   Elective            18869          18.54
                 2                     Urgent            18480          18.16
                 6                        NaN             5291           5.20
                 5              Not Available             4785           4.70
                 8                 Not Mapped              320           0.31
                 7              Trauma Center               21           0.02
                 4                    Newborn               10           0.01

DISCHARGE_DISPOSITION_ID
 discharge_disposition_id                                                                         discharge_disposition_description  encounter_count  encounter_pct
                        1                                                                

In [26]:
# Audit missing descriptions in parsed lookup tables

lookup_description_audit = []

for column, (lookup, description_col) in coded_lookup_pairs.items():

    missing_rows = lookup[lookup[description_col].isna()].copy()

    print("\n" + "=" * 80)
    print(column.upper())
    print("=" * 80)
    print(f"Lookup rows: {len(lookup)}")
    print(f"Missing descriptions: {lookup[description_col].isna().sum()}")

    if not missing_rows.empty:
        print("\nCodes with missing descriptions:")
        print(missing_rows[[column, description_col]].to_string(index=False))

    lookup_description_audit.append(
        {
            "column": column,
            "lookup_rows": len(lookup),
            "missing_descriptions": lookup[description_col].isna().sum(),
        }
    )

lookup_description_audit = pd.DataFrame(lookup_description_audit)

print("\n" + "=" * 80)
print("LOOKUP DESCRIPTION SUMMARY")
print("=" * 80)

lookup_description_audit


ADMISSION_TYPE_ID
Lookup rows: 8
Missing descriptions: 1

Codes with missing descriptions:
admission_type_id admission_type_description
                6                        NaN

DISCHARGE_DISPOSITION_ID
Lookup rows: 31
Missing descriptions: 2

Codes with missing descriptions:
discharge_disposition_id discharge_disposition_description
                      18                               NaN
                     NaN                               NaN

ADMISSION_SOURCE_ID
Lookup rows: 25
Missing descriptions: 1

Codes with missing descriptions:
admission_source_id admission_source_description
                 17                          NaN

LOOKUP DESCRIPTION SUMMARY


,column,lookup_rows,missing_descriptions
0,admission_type_id,8,1
1,discharge_disposition_id,31,2
2,admission_source_id,25,1


In [27]:
# Reload IDS_mapping.csv preserving literal text values
ids_mapping_literal = pd.read_csv(IDS_MAPPING_PATH, keep_default_na=False)

print("Literal mapping file loaded.")
print(f"Rows: {len(ids_mapping_literal):,}")
print(f"Columns: {ids_mapping_literal.shape[1]:,}")

# Find rows containing empty strings or common textual null markers
null_like_markers = {"", "NULL", "null", "NA", "N/A", "NaN", "nan"}

null_like_rows = ids_mapping_literal[
    ids_mapping_literal.apply(
        lambda col: col.astype(str).str.strip().isin(null_like_markers)
    ).any(axis=1)
].copy()

null_like_rows = null_like_rows.reset_index().rename(columns={"index": "row_number"})

print("\nRows containing literal null-like values:")
null_like_rows

Literal mapping file loaded.
Rows: 67
Columns: 2

Rows containing literal null-like values:


,row_number,admission_type_id,description
0,5,6,NULL
1,8,,
2,27,18,NULL
3,40,,
4,57,17,NULL


In [28]:
# Audit encounter usage of lookup codes whose source descriptions are literal "NULL"

null_description_codes = {
    "admission_type_id": [6],
    "discharge_disposition_id": [18],
    "admission_source_id": [17],
}

null_code_usage = []

for column, codes in null_description_codes.items():
    for code in codes:
        encounter_count = df_raw[column].eq(code).sum()
        encounter_pct = encounter_count / len(df_raw) * 100

        null_code_usage.append(
            {
                "column": column,
                "code": code,
                "source_description": "NULL",
                "encounter_count": encounter_count,
                "encounter_pct": round(encounter_pct, 2),
            }
        )

null_code_usage = pd.DataFrame(null_code_usage)

print("Usage of lookup codes with literal NULL descriptions:")
print(
    f"Total affected encounters across code occurrences: "
    f"{null_code_usage['encounter_count'].sum():,}"
)

null_code_usage

Usage of lookup codes with literal NULL descriptions:
Total affected encounters across code occurrences: 15,763


,column,code,source_description,encounter_count,encounter_pct
0,admission_type_id,6,NULL,5291,5.20
1,discharge_disposition_id,18,NULL,3691,3.63
2,admission_source_id,17,NULL,6781,6.66


In [29]:
# Display NULL-description code usage in a compact readable format

print(
    null_code_usage[["column", "code", "encounter_count", "encounter_pct"]].to_string(
        index=False
    )
)

                  column  code  encounter_count  encounter_pct
       admission_type_id     6             5291           5.20
discharge_disposition_id    18             3691           3.63
     admission_source_id    17             6781           6.66


In [30]:
# Count unique encounters affected by at least one NULL-description lookup code

null_code_mask = (
    df_raw["admission_type_id"].eq(6)
    | df_raw["discharge_disposition_id"].eq(18)
    | df_raw["admission_source_id"].eq(17)
)

unique_null_code_encounters = null_code_mask.sum()
unique_null_code_pct = unique_null_code_encounters / len(df_raw) * 100

print(
    f"Unique encounters affected by at least one NULL-description code: "
    f"{unique_null_code_encounters:,}"
)

print(f"Percentage of all encounters affected: " f"{unique_null_code_pct:.2f}%")

Unique encounters affected by at least one NULL-description code: 12,794
Percentage of all encounters affected: 12.57%


In [31]:
# Consolidate key raw-data audit findings

raw_audit_summary = {
    "dataset_rows": len(df_raw),
    "dataset_columns": df_raw.shape[1],
    "unique_encounters": df_raw["encounter_id"].nunique(),
    "unique_patients": df_raw["patient_nbr"].nunique(),
    "duplicate_encounter_ids": df_raw["encounter_id"].duplicated().sum(),
    "exact_duplicate_rows": df_raw.duplicated().sum(),
    "patients_with_multiple_encounters": patients_with_multiple_encounters,
    "max_encounters_per_patient": max_encounters_per_patient,
    "columns_with_identified_missingness": len(missingness_audit),
    "identified_missing_cells": int(missingness_audit["total_missing_count"].sum()),
    "readmitted_within_30d_count": int(positive_count),
    "readmitted_within_30d_pct": round(positive_pct, 2),
    "null_description_code_occurrences": int(null_code_usage["encounter_count"].sum()),
    "unique_encounters_with_null_description_codes": int(unique_null_code_encounters),
    "unique_encounters_with_null_description_codes_pct": round(unique_null_code_pct, 2),
}

raw_audit_summary_df = pd.DataFrame(
    raw_audit_summary.items(), columns=["metric", "value"]
)

raw_audit_summary_df

,metric,value
0,dataset_rows,101766.00
1,dataset_columns,50.00
2,unique_encounters,101766.00
3,unique_patients,71518.00
4,duplicate_encounter_ids,0.00
5,exact_duplicate_rows,0.00
6,patients_with_multiple_encounters,16773.00
7,max_encounters_per_patient,40.00
8,columns_with_identified_missingness,9.00
9,identified_missing_cells,374017.00


In [32]:
# Create a preprocessing decision register from confirmed audit findings

preprocessing_decisions = pd.DataFrame(
    [
        {
            "issue": "Question-mark missing values",
            "affected_fields": "weight, medical_specialty, payer_code, race, diag_1, diag_2, diag_3",
            "decision": "Convert '?' to explicit missing values during preprocessing",
            "rationale": "Question marks are source-level missing-value markers, not valid categories.",
        },
        {
            "issue": "Native null values",
            "affected_fields": "max_glu_serum, A1Cresult",
            "decision": "Preserve initially; evaluate semantic meaning before encoding",
            "rationale": "Native nulls may represent tests not performed rather than ordinary missing data.",
        },
        {
            "issue": "Extremely sparse fields",
            "affected_fields": "weight, max_glu_serum, A1Cresult",
            "decision": "Evaluate for exclusion or clinically meaningful missingness indicators",
            "rationale": "Very high missingness can make direct imputation unreliable.",
        },
        {
            "issue": "Repeated patients",
            "affected_fields": "patient_nbr",
            "decision": "Use patient-aware train/test splitting for predictive modeling",
            "rationale": "The same patient appearing in both training and test sets could cause information leakage.",
        },
        {
            "issue": "Lookup codes with literal NULL descriptions",
            "affected_fields": "admission_type_id, discharge_disposition_id, admission_source_id",
            "decision": "Retain codes and label descriptions as source-defined unknown/NULL",
            "rationale": "Valid encounter codes exist even though source metadata provides no descriptive label.",
        },
        {
            "issue": "Readmission target",
            "affected_fields": "readmitted",
            "decision": "Create binary target: <30 = 1; NO and >30 = 0",
            "rationale": "Project objective is prediction of readmission within 30 days.",
        },
        {
            "issue": "Class imbalance",
            "affected_fields": "readmitted",
            "decision": "Use imbalance-aware evaluation metrics",
            "rationale": "Only 11.16% of encounters are positive; accuracy alone would be misleading.",
        },
        {
            "issue": "Encounter identifier",
            "affected_fields": "encounter_id",
            "decision": "Retain for lineage but exclude from model features",
            "rationale": "Identifier has no intended predictive meaning.",
        },
        {
            "issue": "Patient identifier",
            "affected_fields": "patient_nbr",
            "decision": "Retain for grouping/splitting but exclude as a direct model feature",
            "rationale": "Patient identity should not become a memorized predictive signal.",
        },
    ]
)

pd.set_option("display.max_colwidth", None)

preprocessing_decisions

,issue,affected_fields,decision,rationale
0,Question-mark missing values,"weight, medical_specialty, payer_code, race, diag_1, diag_2, diag_3",Convert '?' to explicit missing values during preprocessing,"Question marks are source-level missing-value markers, not valid categories."
1,Native null values,"max_glu_serum, A1Cresult",Preserve initially; evaluate semantic meaning before encoding,Native nulls may represent tests not performed rather than ordinary missing data.
2,Extremely sparse fields,"weight, max_glu_serum, A1Cresult",Evaluate for exclusion or clinically meaningful missingness indicators,Very high missingness can make direct imputation unreliable.
3,Repeated patients,patient_nbr,Use patient-aware train/test splitting for predictive modeling,The same patient appearing in both training and test sets could cause information leakage.
4,Lookup codes with literal NULL descriptions,"admission_type_id, discharge_disposition_id, admission_source_id",Retain codes and label descriptions as source-defined unknown/NULL,Valid encounter codes exist even though source metadata provides no descriptive label.
5,Readmission target,readmitted,Create binary target: <30 = 1; NO and >30 = 0,Project objective is prediction of readmission within 30 days.
6,Class imbalance,readmitted,Use imbalance-aware evaluation metrics,Only 11.16% of encounters are positive; accuracy alone would be misleading.
7,Encounter identifier,encounter_id,Retain for lineage but exclude from model features,Identifier has no intended predictive meaning.
8,Patient identifier,patient_nbr,Retain for grouping/splitting but exclude as a direct model feature,Patient identity should not become a memorized predictive signal.


## Raw Data Audit — Key Conclusions

The raw dataset contains 101,766 hospital encounters representing 71,518 unique patients. Encounter identifiers are unique, and no exact duplicate records were detected.

A substantial repeated-patient structure is present: 16,773 patients have multiple encounters, and one patient appears in as many as 40 encounters. This requires patient-aware partitioning during predictive modeling to reduce the risk of information leakage across training and evaluation datasets.

Missing information is represented through multiple mechanisms. Seven variables use the literal `?` marker, while `max_glu_serum` and `A1Cresult` contain native null values. Across the dataset, 374,017 missing cells were identified in nine variables. Missingness is particularly severe for `weight`, `max_glu_serum`, and `A1Cresult`, so these fields require explicit treatment rather than routine imputation.

The 30-day readmission outcome is imbalanced. Only 11,357 encounters (11.16%) are followed by readmission within 30 days. A classifier predicting the majority class for every encounter would achieve approximately 88.84% accuracy, demonstrating that accuracy alone will be an inadequate model-performance measure.

The three administrative code fields — admission type, discharge disposition, and admission source — have complete code coverage in the supplied reference mapping. However, the source metadata contains literal `NULL` descriptions for several valid codes. These codes occur in 12,794 unique encounters (12.57% of the dataset) and will therefore be retained as source-defined unknown categories rather than treated as corrupted observations.

Two medication variables, `examide` and `citoglipton`, are constant across all encounters and provide no variance for downstream modeling. Several additional medication variables are extremely sparse and will require further evaluation before feature inclusion.

This audit establishes the requirements for the downstream preprocessing, PostgreSQL schema, statistical analysis, and predictive-modeling pipeline. No modifications have been made to the files under `data/raw/`.